In [24]:
# build shallow network carbapnemase classifier in pytorch
#logistic regression carbapenemase classifier
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import random
import json
from copy import deepcopy
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# DEVICE = "mps" #mac graphics card
EMB_TYPE = "MeanEmbedding"  # or "CLSEmbedding"
warnings.filterwarnings("ignore")
print("DEVICE:", DEVICE)
print("EMB_TYPE:", EMB_TYPE)

DEVICE: cpu
EMB_TYPE: MeanEmbedding


In [25]:
df = pd.read_csv("embeddings.csv")
df.head()

,Family,Gene,Sequence,Carbapenemase,MeanEmbedding,CLSEmbedding,EmbeddingDim
0,SHV,SHV-52,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,0,"[[0.05962217226624489, -0.07687947154045105, 0...","[[0.07300693541765213, -0.0206560418009758, 0....",1280
1,CTX-M,CTX-M-130,MVTKRVQRMMFAAAACIPLLLGSAPLYAQTSAVQQKLAALEKSSGG...,0,"[[0.05877629667520523, -0.06891713291406631, -...","[[0.10748688131570816, -0.044926486909389496, ...",1280
2,TEM,TEM-126,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,0,"[[0.059695929288864136, -0.05530138313770294, ...","[[0.07826980203390121, -0.01159658282995224, 0...",1280
3,TEM,TEM-72,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0,"[[0.056167904287576675, -0.04905174672603607, ...","[[0.07821495085954666, -0.010604368522763252, ...",1280
4,TEM,TEM-59,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0,"[[0.07185406237840652, -0.056960783898830414, ...","[[0.08561597019433975, -0.01823832094669342, 0...",1280


In [26]:
#prepare dataset:
X = np.vstack(df[EMB_TYPE].apply(lambda x: np.fromstring(x.strip("[]"), sep=" ")))
y = df["Carbapenemase"].values

#scale features
X_scaled = StandardScaler().fit_transform(X)

In [27]:
# shallow fully connected network
class CarbapenemaseMLP(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.net = nn.Sequential(
            #nn.Linear(emb_dim, 1024),
            #nn.GELU(),
            #nn.Dropout(0.2),
            #nn.Linear(emb_dim, 512),
            #nn.GELU(),
            #nn.Dropout(0.2),
            nn.Linear(emb_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # returns logits

In [28]:
# training function
def train_one_fold(model, X_train, y_train, epochs=50, lr=1e-4, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    return model

In [29]:
# prediction function
def predict_probs(model, X):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()

    return probs

In [30]:
# 5 fold cv
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

metrics = {
    "fold": [],
    "accuracy": [],
    "balanced_accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": [],
    "avg_precision": [],
    "threshold": []
}

best_threshold = 0.5  # replace with your tuned threshold if desired

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit scaler on training fold only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build model
    model = CarbapenemaseMLP(emb_dim=X_train_scaled.shape[1]) 

    model = train_one_fold(
        model,
        X_train_scaled,
        y_train,
        epochs=50,
        batch_size=32,
    )

    # Predict
    y_prob = predict_probs(model, X_test_scaled)
    y_pred = (y_prob >= best_threshold).astype(int)

    # Metrics
    metrics["fold"].append(fold)
    metrics["accuracy"].append(accuracy_score(y_test, y_pred))
    metrics["balanced_accuracy"].append(balanced_accuracy_score(y_test, y_pred))
    metrics["precision"].append(precision_score(y_test, y_pred, zero_division=0))
    metrics["recall"].append(recall_score(y_test, y_pred, zero_division=0))
    metrics["f1"].append(f1_score(y_test, y_pred, zero_division=0))
    metrics["roc_auc"].append(roc_auc_score(y_test, y_prob))
    metrics["avg_precision"].append(average_precision_score(y_test, y_prob))
    metrics["threshold"].append(best_threshold)

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

   fold  accuracy  balanced_accuracy  precision    recall        f1   roc_auc  \
0     1  0.650000           0.726091   0.418182  0.884615  0.567901  0.833420   
1     2  0.715000           0.782484   0.475248  0.923077  0.627451  0.889553   
2     3  0.675000           0.754781   0.445455  0.924528  0.601227  0.835451   
3     4  0.648241           0.699765   0.411765  0.807692  0.545455  0.798012   
4     5  0.713568           0.775052   0.474747  0.903846  0.622517  0.862507   

   avg_precision  threshold  
0       0.683748        0.5  
1       0.827824        0.5  
2       0.594402        0.5  
3       0.649367        0.5  
4       0.720294        0.5  


In [31]:
# train final model on all data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
final_model = CarbapenemaseMLP(emb_dim=X_scaled.shape[1])
final_model = train_one_fold(
    final_model,
    X_scaled,
    y,
    epochs=50,
    batch_size=32,
)
# save final model and scaler
torch.save(final_model.state_dict(), "final_carbapenemase_mlp_fcn.pth")
with open("scaler_params_fcn.json", "w") as f:
    json.dump({
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist()
    }, f)

#save final model metrics
final_metrics = {
    "accuracy": metrics_df["accuracy"].mean(),
    "balanced_accuracy": metrics_df["balanced_accuracy"].mean(),
    "precision": metrics_df["precision"].mean(),
    "recall": metrics_df["recall"].mean(),
    "f1": metrics_df["f1"].mean(),
    "roc_auc": metrics_df["roc_auc"].mean(),
    "avg_precision": metrics_df["avg_precision"].mean(),
    "threshold": best_threshold
}

#save final metrics to csv
final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df.to_csv("final_model_metrics_fcn.csv", index=False)
final_metrics_df

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,avg_precision,threshold
0,0.680362,0.747635,0.445079,0.888752,0.59291,0.843788,0.695127,0.5


# problem is non-carbapenemases are getting called as carbapenemases, need to incorporate non class-A carbapenamases